### We want to calculate the smallest $ \eta $ for our experiments to see if it is smaller than our grid cell size $ \Delta x, \Delta z $

In [1]:
#import packages

using Oceananigans
using CairoMakie
using NCDatasets
using Statistics
using Printf


In [ ]:
#import datasets we need the oceanostics file for the dissipation rate
# we will analyze the 1e5, 1e6, and 1e7 Ra cases with hill

ds_1 = NCDataset(string("/Users/hfdrake/code/HorizontalConvection/output/turbulent_onehill_0.6_Ra100000.0_coldstart_oceanostics.nc"));
ds_2 = NCDataset(string("/Users/hfdrake/code/HorizontalConvection/output/turbulent_onehill_0.6_Ra1.0e6_coldstart_oceanostics.nc"));
ds_3 = NCDataset(string("/Users/hfdrake/code/HorizontalConvection/output/turbulent_onehill_0.6_Ra5.0e6_coldstart_oceanostics.nc"));
ds_4 = NCDataset(string("/Users/hfdrake/code/HorizontalConvection/output/turbulent_onehill_0.6_Ra1.0e7_coldstart_oceanostics.nc"));

list_hill_datasets = [ds_1, ds_2, ds_3, ds_4];


In [35]:
#define the grid cell sizes Δx is always gonna be larger than Δz so we only need to check Δz

Δz_1 = 1/169
Δz_2 = 1/301
Δz_3 = 1/450
Δz_4 = 1/535;

Δzs = [Δz_1, Δz_2, Δz_3, Δz_4];

In [37]:
ε_1 = ds_1["ε"][4+1:end-4, 1, 4+1:end-4, :]
ε_2 = ds_2["ε"][4+1:end-4, 1, 4+1:end-4, :]
ε_3 = ds_3["ε"][4+1:end-4, 1, 4+1:end-4, :]
ε_4 = ds_4["ε"][4+1:end-4, 1, 4+1:end-4, :];

epsilons = [ε_1, ε_2, ε_3, ε_4];

In [40]:
function get_η(ε)
    η = ε.^(-1/4)
    return η
end

get_η (generic function with 1 method)

In [45]:
η_1 = get_η(ε_1)
η_min_1 = minimum(x for x in η_1 if x > 0)
if Δz_1 < η_min_1
    @printf("The Kolmogorov scale is resolved for Ra = 1e5 with Δz = %.5f and η_min = %.5f\n", Δz_1, η_min_1)
else
    @printf("The Kolmogorov scale is NOT resolved for Ra = 1e5 with Δz = %.5f and η_min = %.5f\n", Δz_1, η_min_1)
end

The Kolmogorov scale is resolved for Ra = 1e5 with Δz = 0.00592 and η_min = 1.72137


In [46]:
function is_kolmogorov_resolved(Δz, ε)
    η = get_η(ε)
    η_min = minimum(x for x in η if x > 0)
    if Δz < η_min
        return true, η_min
    else
        return false, η_min
    end
end

is_kolmogorov_resolved (generic function with 1 method)

In [51]:
for (i,j) in zip(epsilons, Δzs)
    resolved, η_min = is_kolmogorov_resolved(j, i)
    if resolved
        @printf("The Kolmogorov scale is resolved with Δz = %.5f and η_min = %.5f\n", j, η_min)
    else
        @printf("The Kolmogorov scale is NOT resolved with Δz = %.5f and η_min = %.5f\n", j, η_min)
    end
end

The Kolmogorov scale is resolved with Δz = 0.00592 and η_min = 1.72137
The Kolmogorov scale is resolved with Δz = 0.00332 and η_min = 1.52066
The Kolmogorov scale is resolved with Δz = 0.00222 and η_min = 1.37774
The Kolmogorov scale is resolved with Δz = 0.00187 and η_min = 1.17849
